# prep.tableS2

Table S2 contains the data of habitat diversity for virus, bacteria, plants, and hosts, and the number of cooccurrences per habitat. 

In [1]:
import pandas as pd
from daforfer import DaforferDB
from skbio.diversity.alpha import chao1
import networkx as nx
from yaml import load, Loader
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
si.toc()

┌─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  name   │                                                        description                                                        │
│ varchar │                                                          varchar                                                          │
├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ TableS1 │ Table S1: Library sites and context                                                                                       │
│ TableS2 │ This table summarizes most of the information of our detected OTUs, including host_range, site_range, habitat_range, etc. │
│ TableS3 │ Site-level diversity and number of cooccurring virus-bacteria                                                             │
│ TableS4 │ Habitat-level diversity and number o

## PAB diversity

In [2]:
metadata = db.conn.sql('SELECT * FROM D_sites').df()
bacteria_hits = db.conn.sql('SELECT * FROM D_PABHits').df()
bacteria_hits = pd.merge(metadata, bacteria_hits, on='library', how='left')
bacteria_hits

,site,library,habitat,n_extracts,host_taxon,taxid,scientific_name,is_pab,pab_type
0,C1,PV534,Crop,3,Diplotaxis erucoides,NaN,NaN,NaN,NaN
1,C1,PV535,Crop,17,Brassica oleracea,1563157.0,Pseudomonas endophytica,True,pab_unknown
2,C1,PV535,Crop,17,Brassica oleracea,1270.0,Micrococcus luteus,True,pab_unknown
3,C1,PV538,Crop,8,Brassica oleracea,NaN,NaN,NaN,NaN
4,C1,PV540,Crop,1,Picris echioides,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
626,Z2,PV527,Crop,4,Convolvulus arvensis,1770058.0,Devosia elaeis,True,pab_unknown
627,Z2,PV529,Crop,1,Picris echioides,47880.0,Pseudomonas fulva,True,pab_unknown
628,Z2,PV529,Crop,1,Picris echioides,1220495.0,Pseudomonas punonensis,True,pab_unknown
629,Z2,PV529,Crop,1,Picris echioides,289370.0,Pseudomonas argentinensis,True,pab_unknown


We simply create a list of item counts in one of the columns. 

In [3]:
pab_alpha_diversity = bacteria_hits.value_counts(
    ['habitat', 'scientific_name']
    ).reset_index().groupby(
        ['habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
pab_alpha_diversity

,habitat,hits
0,Crop,"[7, 5, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ..."
1,Edge,"[15, 13, 10, 10, 6, 6, 6, 5, 4, 4, 4, 4, 4, 4,..."
2,Oak,"[9, 4, 3, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, ..."
3,Wasteland,"[11, 10, 8, 6, 6, 5, 5, 4, 4, 3, 3, 2, 2, 2, 2..."


In [4]:
pab_alpha_diversity['species_richness'] = pab_alpha_diversity['hits'].apply(lambda x: len(x))
pab_alpha_diversity = pab_alpha_diversity.sort_values(by=['habitat'])
pab_alpha_diversity['disturbed'] = pab_alpha_diversity['habitat'].apply(
    lambda x: {"Crop": "disturbed", "Wasteland": "non-disturbed", "Edge": "disturbed", "Oak": "non-disturbed"}[x]
)
pab_alpha_diversity = pab_alpha_diversity.drop(columns=['hits'])[['habitat', 'disturbed', 'species_richness']]
pab_alpha_diversity

,habitat,disturbed,species_richness
0,Crop,disturbed,52
1,Edge,disturbed,77
2,Oak,non-disturbed,26
3,Wasteland,non-disturbed,61


## Virus diversity

In [5]:
# virus_hits = pd.read_csv("output/hits.virus.csv", sep=";")
virus_hits = db.conn.sql('SELECT * FROM D_virusHits').df()
virus_hits = pd.merge(metadata, virus_hits, on='library', how='left')
virus_alpha_diversity = virus_hits.value_counts(
    ['habitat', 'scientific_name']
    ).reset_index().groupby(
        ['habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
virus_alpha_diversity['species_richness'] = virus_alpha_diversity['hits'].apply(lambda x: len(x))
virus_alpha_diversity = virus_alpha_diversity.drop(columns=['hits'])[['habitat', 'species_richness']]
virus_alpha_diversity


,habitat,species_richness
0,Crop,64
1,Edge,113
2,Oak,39
3,Wasteland,59


## Plant diversity

In [6]:

plant_hits = pd.read_csv("input/plant-data-3.csv", sep=';')
plant_hits = pd.merge(
    plant_hits, metadata[['site', 'habitat']].drop_duplicates('site'), on='site', how='inner'
)
plant_alpha_diversity = plant_hits.drop_duplicates(
    ['habitat', 'species name'], keep='first'
).value_counts('habitat').reset_index().sort_values('habitat').rename(columns={'count': 'species_richness'})
plant_alpha_diversity

,habitat,species_richness
3,Crop,63
2,Edge,103
1,Oak,117
0,Wasteland,124


## Host diversity

In [7]:
host_alpha_diversity = metadata.value_counts(
    ['habitat', 'host_taxon']
).reset_index().groupby(
    ['habitat']
)['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
host_alpha_diversity['species_richness'] = host_alpha_diversity['hits'].apply(lambda x: len(x))
host_alpha_diversity = host_alpha_diversity.drop(columns=['hits'])[['habitat',  'species_richness']]
host_alpha_diversity

,habitat,species_richness
0,Crop,28
1,Edge,52
2,Oak,51
3,Wasteland,52


## Merge

In [8]:
# bacteria_alpha_diversity = db.conn.query('SELECT * FROM D_PABAlphaDiv').df()
alpha_diversity = pd.merge(
    pab_alpha_diversity[['habitat', 'disturbed', 'species_richness']], 
    virus_alpha_diversity[['habitat', 'species_richness']],
    on='habitat', suffixes=['_bact', '_vir']
)

alpha_diversity = pd.merge(
    alpha_diversity,
    plant_alpha_diversity[['habitat', 'species_richness']].rename(
        columns={
            'species_richness': 'species_richness_plant',
        }
    ),
    on='habitat'
)

alpha_diversity = pd.merge(
    alpha_diversity,
    host_alpha_diversity[['habitat', 'species_richness']].rename(
        columns={
            'species_richness': 'species_richness_host',
        }
    ),
    on='habitat'
)


# alpha_diversity.to_csv("output/diversity.all.csv", sep=";", index=False)
# db.save_dataframe(
#     df=alpha_diversity, table_name="D_ADAllOrganismsSite",
#     description="Alpha diversity per site for virus, plant, host and bacteria at site level"
# )

alpha_diversity

,habitat,disturbed,species_richness_bact,species_richness_vir,species_richness_plant,species_richness_host
0,Crop,disturbed,52,64,63,28
1,Edge,disturbed,77,113,103,52
2,Oak,non-disturbed,26,39,117,51
3,Wasteland,non-disturbed,61,59,124,52


## Cooccurrences by site

In [9]:
cooccurrence_network = nx.read_graphml("output/network.coocurrence.virusbact-bylibrary.trans.graphml")
cooccurrence_network.number_of_edges()

59

In [10]:
cooccurrence_codetections_by_library = db.conn.sql('SELECT * FROM D_coocDetections').df()
cooccurrence_codetections_by_habitat = pd.merge(metadata, cooccurrence_codetections_by_library, on='library').groupby('habitat', as_index=False)['number_cooccurrences_per_library'].sum()
cooccurrence_codetections_by_habitat = cooccurrence_codetections_by_habitat.rename(columns={'number_cooccurrences_per_library': 'coccurrence_codetections'}) # type: ignore
cooccurrence_codetections_by_habitat['coccurrence_codetections'] = cooccurrence_codetections_by_habitat['coccurrence_codetections'].astype(int)
cooccurrence_codetections_by_habitat

,habitat,coccurrence_codetections
0,Crop,34
1,Edge,267
2,Oak,11
3,Wasteland,33


In [11]:
cooccurrence_detection_pairs_by_library = db.conn.sql('SELECT * FROM D_coocPairDetections').df()
cooccurrence_detections_by_habitat = pd.merge(metadata, cooccurrence_detection_pairs_by_library, on='library').drop_duplicates(
    ['pair', 'habitat']
)[['habitat', 'pair']].value_counts(['habitat']).reset_index().rename(columns={'count': 'cooccurrence_detections'})
cooccurrence_detections_by_habitat

,habitat,cooccurrence_detections
0,Edge,51
1,Crop,22
2,Wasteland,15
3,Oak,7


In [12]:
tableS4 = pd.merge(alpha_diversity, cooccurrence_codetections_by_habitat, on=['habitat'])
tableS4 = pd.merge(tableS4, cooccurrence_detections_by_habitat, on=['habitat'])
tableS4

,habitat,disturbed,species_richness_bact,species_richness_vir,species_richness_plant,species_richness_host,coccurrence_codetections,cooccurrence_detections
0,Crop,disturbed,52,64,63,28,34,22
1,Edge,disturbed,77,113,103,52,267,51
2,Oak,non-disturbed,26,39,117,51,11,7
3,Wasteland,non-disturbed,61,59,124,52,33,15


## Save

In [13]:
db.save_dataframe(
    tableS4, "D_Habitat_level_div",
    description="Habitat-level diversity and number of cooccurring virus-bacteria"
)

Saved D_Habitat_level_div to db.2025-11-17


In [14]:
si.save_dataframe(
    tableS4, "TableS4",
    description="Habitat-level diversity and number of cooccurring virus-bacteria"
)

Saved TableS4 to si.2025-11-17


In [15]:
tableS4.to_csv("output/TableS4.csv", sep=";")

In [16]:
si.conn.close()
db.conn.close()